In [ ]:
import random
import time

import cv2
import face_recognition
import matplotlib.pyplot as plt
import numpy as np

# --- Paramètres ---
INDEX_CAMERA = 1      # 0 = webcam intégrée, 1 = caméra externe
TOLERANCE = 0.6       # distance max entre deux visages de la même personne
SEUIL_FACE = 0.10     # |rotation| max pour être considéré de face
SEUIL_PROFIL = 0.15   # |rotation| min pour être considéré de profil (tête tournée ~30-45°)
DUREE_STABLE = 1.0    # secondes de pose correcte et stable avant la prise automatique
MAX_TENTATIVES = 3    # nombre de séries autorisées avant abandon

# Consignes affichées dans la fenêtre caméra (OpenCV n'affiche pas les accents)
POSES = {
    "face": "FACE : regardez la camera",
    "gauche": "PROFIL GAUCHE : tournez la tete vers votre gauche",
    "droite": "PROFIL DROIT : tournez la tete vers votre droite",
}
LIBELLES = {"face": "Face", "gauche": "Profil gauche", "droite": "Profil droit"}

In [ ]:
def mesurer_rotation(landmarks):
    """
    Estime la rotation horizontale de la tête à partir des repères du visage :
    position du bout du nez entre les deux bords de la mâchoire.
    ~0 de face ; > 0 tête tournée vers SA gauche ; < 0 vers SA droite (image non miroir).
    """
    machoire = landmarks["chin"]
    x_bord_gauche, x_bord_droit = machoire[0][0], machoire[-1][0]
    largeur = x_bord_droit - x_bord_gauche
    if largeur <= 0:
        return None
    x_nez = np.mean([p[0] for p in landmarks["nose_tip"]])
    return (x_nez - x_bord_gauche) / largeur - 0.5


def pose_detectee(rotation):
    if rotation is None:
        return None
    if abs(rotation) <= SEUIL_FACE:
        return "face"
    if rotation >= SEUIL_PROFIL:
        return "gauche"
    if rotation <= -SEUIL_PROFIL:
        return "droite"
    return None  # entre les deux : ni de face, ni assez tourné


def analyser_photo(image, pose_attendue):
    """
    Vérifie une photo : un seul visage, dans la pose attendue.
    Retourne un dict {ok, message, emplacement, rotation}.
    """
    emplacements = face_recognition.face_locations(image)
    if not emplacements:
        return {"ok": False, "message": "aucun visage détecté (tête trop tournée, trop loin ou mal éclairée ?)"}
    if len(emplacements) > 1:
        return {"ok": False, "message": f"{len(emplacements)} visages détectés, une seule personne attendue"}

    landmarks = face_recognition.face_landmarks(image, emplacements)[0]
    rotation = mesurer_rotation(landmarks)
    pose = pose_detectee(rotation)
    if pose != pose_attendue:
        vue = LIBELLES.get(pose, "pose intermédiaire")
        return {"ok": False, "rotation": rotation,
                "message": f"pose attendue « {LIBELLES[pose_attendue]} », détectée « {vue} »"}

    return {"ok": True, "message": "OK", "emplacement": emplacements[0], "rotation": rotation}


def valider_serie(photos):
    """
    Valide une série {pose: image} : les 3 poses doivent être correctes
    ET les 3 photos doivent montrer la même personne.
    Retourne (ok, erreurs, encodages).
    """
    erreurs, encodages = [], {}
    for pose in POSES:
        image = photos.get(pose)
        if image is None:
            erreurs.append(f"{LIBELLES[pose]} : photo manquante")
            continue
        analyse = analyser_photo(image, pose)
        if not analyse["ok"]:
            erreurs.append(f"{LIBELLES[pose]} : {analyse['message']}")
            continue
        encodages[pose] = face_recognition.face_encodings(image, [analyse["emplacement"]], num_jitters=5)[0]

    # Cohérence interne : les profils doivent être la même personne que la face
    if not erreurs:
        for pose in ("gauche", "droite"):
            distance = face_recognition.face_distance([encodages["face"]], encodages[pose])[0]
            if distance > TOLERANCE:
                erreurs.append(f"{LIBELLES[pose]} : pas la même personne que la photo de face (distance {distance:.2f})")

    return not erreurs, erreurs, encodages

In [ ]:
def ecrire(image, texte, ligne, couleur):
    cv2.putText(image, texte, (10, 30 * ligne), cv2.FONT_HERSHEY_SIMPLEX, 0.6, couleur, 2)


def capturer_pose(cap, pose, titre):
    """
    Affiche le flux caméra avec la consigne. La photo est prise AUTOMATIQUEMENT
    dès que la pose est correcte et tenue pendant DUREE_STABLE secondes.
    Q/ECHAP annule. Retourne l'image RGB capturée (non miroir) ou None.
    """
    fenetre = f"{titre} - prise automatique | Q/ECHAP : annuler"
    debut_pose_ok = None  # instant où la pose est devenue correcte

    while True:
        ok, frame = cap.read()
        if not ok:
            print("❌ Impossible de lire une image depuis la caméra.")
            return None

        # Analyse rapide sur une image réduite pour suivre la pose en direct
        petite = cv2.cvtColor(cv2.resize(frame, (0, 0), fx=0.5, fy=0.5), cv2.COLOR_BGR2RGB)
        pose_ok = analyser_photo(petite, pose)["ok"]

        if not pose_ok:
            debut_pose_ok = None
        elif debut_pose_ok is None:
            debut_pose_ok = time.monotonic()

        affichage = cv2.flip(frame, 1)  # effet miroir : plus naturel pour se placer
        couleur = (0, 200, 0) if pose_ok else (0, 0, 255)
        ecrire(affichage, POSES[pose], 1, couleur)

        if pose_ok:
            restant = DUREE_STABLE - (time.monotonic() - debut_pose_ok)
            if restant <= 0:
                # Contrôle final sur l'image pleine résolution avant de garder la photo
                image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                if analyser_photo(image, pose)["ok"]:
                    ecrire(affichage, "Photo prise !", 2, couleur)
                    cv2.imshow(fenetre, affichage)
                    cv2.waitKey(700)  # court retour visuel avant la pose suivante
                    return image
                debut_pose_ok = None  # pose perdue en pleine résolution : on recommence
            else:
                ecrire(affichage, f"Ne bougez plus... {restant:.1f}s", 2, couleur)
        else:
            ecrire(affichage, "Pose incorrecte", 2, couleur)

        cv2.imshow(fenetre, affichage)
        touche = cv2.waitKey(1) & 0xFF
        if touche in (27, ord("q")):          # ECHAP / q -> annule
            return None


def capturer_serie(titre, ordre_aleatoire=False):
    """
    Fait prendre les 3 photos (face, profil gauche, profil droit) puis les valide.
    Si une seule photo est incorrecte, TOUTE la série est à reprendre.
    ordre_aleatoire : les poses sont demandées dans un ordre imprévisible,
    ce qui complique le rejeu d'une vidéo préenregistrée.
    Retourne (photos, encodages), ou (None, None) en cas d'abandon / échec.
    """
    for tentative in range(1, MAX_TENTATIVES + 1):
        ordre = list(POSES)
        if ordre_aleatoire:
            random.shuffle(ordre)
        print(f"📷 {titre} — série {tentative}/{MAX_TENTATIVES} : "
              + " → ".join(LIBELLES[p] for p in ordre))

        cap = cv2.VideoCapture(INDEX_CAMERA, cv2.CAP_DSHOW)
        if not cap.isOpened():
            print(f"❌ Impossible d'ouvrir la caméra (index {INDEX_CAMERA}).")
            return None, None

        photos = {}
        try:
            for pose in ordre:
                image = capturer_pose(cap, pose, titre)
                if image is None:
                    print("❌ Capture annulée.")
                    return None, None
                photos[pose] = image
        finally:
            cap.release()
            cv2.destroyAllWindows()
            cv2.waitKey(1)  # laisse Windows fermer proprement la fenêtre

        ok, erreurs, encodages = valider_serie(photos)
        if ok:
            print("✅ Série validée.")
            return photos, encodages

        print("❌ Série refusée, il faut reprendre les 3 photos :")
        for erreur in erreurs:
            print("   -", erreur)

    print(f"⛔ Échec après {MAX_TENTATIVES} séries.")
    return None, None

In [ ]:
# --- 1re série : photo d'identité (référence) ---
photos_identite, encodages_identite = capturer_serie("Identite")

In [ ]:
# --- 2e série : vérification (ordre des poses aléatoire) ---
photos_verif, encodages_verif = capturer_serie("Verification", ordre_aleatoire=True)

In [ ]:
def comparer_series(encodages_identite, encodages_verif, tolerance=TOLERANCE):
    """
    Compare pose par pose (face/face, gauche/gauche, droite/droite).
    Match uniquement si les 3 poses correspondent.
    """
    if not encodages_identite or not encodages_verif:
        return {"succes": False, "message": "Il manque une série de photos validée."}

    distances = {
        pose: round(float(face_recognition.face_distance([encodages_identite[pose]], encodages_verif[pose])[0]), 4)
        for pose in POSES
    }
    is_match = all(d <= tolerance for d in distances.values())

    # Score intuitif sur la distance moyenne (0.6 de distance = 50 %)
    distance_moyenne = float(np.mean(list(distances.values())))
    score_similitude = max(0.0, (1 - distance_moyenne / (tolerance * 2)) * 100)

    return {
        "succes": True,
        "is_match": is_match,
        "distances": distances,
        "distance_moyenne": round(distance_moyenne, 4),
        "score_pourcentage": round(score_similitude, 2),
    }

resultats = comparer_series(encodages_identite, encodages_verif)
print(resultats)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for ligne, (nom_serie, photos) in enumerate([("Identité", photos_identite), ("Vérification", photos_verif)]):
    for col, pose in enumerate(POSES):
        ax = axes[ligne][col]
        if photos is not None:
            ax.imshow(photos[pose])
        titre = f"{nom_serie} — {LIBELLES[pose]}"
        if ligne == 1 and resultats["succes"]:
            titre += f"\ndistance {resultats['distances'][pose]:.2f}"
        ax.set_title(titre)
        ax.axis('off')

if resultats["succes"]:
    couleur = 'green' if resultats["is_match"] else 'red'
    verdict = "MÊME PERSONNE" if resultats["is_match"] else "PERSONNES DIFFÉRENTES"
    titre = (f"Verdict : {verdict}\nScore de similitude : {resultats['score_pourcentage']:.2f}% "
             f"(distance moyenne : {resultats['distance_moyenne']:.2f})")
else:
    couleur = 'orange'
    titre = f"Analyse impossible : {resultats['message']}"

plt.suptitle(titre, fontsize=14, color=couleur, fontweight='bold')
plt.tight_layout()
plt.show()